# Importação de Bibliotecas

In [1]:
# --- Configuração do Navegador ---
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager # Gerencia o driver automaticamente

# --- Navegação e Interação ---
from selenium.webdriver.common.by import By  # O "GPS": acha botões, caixas de texto, links

# --- Espera Inteligente (Essencial para não dar erro) ---
from selenium.webdriver.support.ui import WebDriverWait # O "cronômetro" de espera
from selenium.webdriver.support import expected_conditions # As condições (ex: "espere até o botão aparecer")

# --- Manipulação de Arquivos e Pastas ---
import os      # Comandos do sistema (criar pastas, verificar caminhos)
import glob    # Lista arquivos na pasta (ex: pegar o último arquivo baixado)
import shutil  # Move e copia arquivos (ex: tirar de Downloads e mover para a pasta final)

# --- Ferramentas Gerais ---
import time # Para pausas fixas
import pandas as pd # Para manipular as tabelas de dados

# Configurações gerais

In [2]:
# Pasta Downloads do usuário
pasta_downloads_padrao = os.path.join(os.path.expanduser("~"), "Downloads")
# Caminho para pasta dos resumos
pasta_destino_projeto = os.path.join('../dados/bacen/', 'resumo_bacen')
# Se a pasta do resumo não tiver criada, criar
if not os.path.exists(pasta_destino_projeto):
    os.makedirs(pasta_destino_projeto)

# Configurações do Navegador

In [3]:
# Serviço de instalação/atualização do driver Chrome
servico = Service(ChromeDriverManager().install())
# Abrir Navegador
navegador = webdriver.Chrome(service=servico)
# Maximizar tela
navegador.maximize_window()
# Abrir Site do Bacen (IF.DATA)
navegador.get("https://www3.bcb.gov.br/ifdata/")

# Função mestre *(Navegação, Cliques, Download e Mover Arquivos)*

In [4]:
def baixar_uma_data(driver, texto_data):
    """
    Função que recebe o navegador e uma string de data (ex: '03/2022'),
    faz todo o processo de clique, download e movimentação do arquivo.
    """

    # Converter data em data
    data_dt = pd.to_datetime(texto_data, format='%m/%Y', errors='coerce')

    ano = data_dt.year
    
    # Selecionar Data
    driver.find_element(By.ID, 'btnDataBase').click()
    print("Campo data selecionado")

    # Espera inteligente pela lista de datas
    if ano <= 2024:
        print("Aguardando lista de datas...")
        esperar = WebDriverWait(navegador,20)
        # USa CSS_SELECTOR para procurar por "li" DENTRO de "#ulDataBase"
        # O comando abaixo espera até que o primeiro item da lista seja visível
        esperar.until(expected_conditions.visibility_of_element_located((By.CSS_SELECTOR, "#ulDataBase li")))
    else:
        time.sleep(0.5)

    # Selecionar data
    driver.find_element(By.LINK_TEXT, texto_data).click()
    print("Data selecionada no menu.")

    # Scroll preventivo
    driver.execute_script("window.scrollTo(0, 0);")
    time.sleep(0.5)

    # Selecionar Instituição
    driver.find_element(By.ID, 'btnTipoInst').click()
    time.sleep(0.5)
    driver.find_element(By.LINK_TEXT, 'Instituições Individuais').click()
    print("Instituição selecionada no menu.")

    # Selecionar Relatório
    driver.find_element(By.ID, 'btnRelatorio').click()
    time.sleep(0.5)
    driver.find_element(By.LINK_TEXT, 'Resumo').click()
    print("Relatório selecionado no menu.")

    # Baixar CSV
    print("Aguardando botão de download...")
    # Espero inteligente pelo botão de exportar
    wait = WebDriverWait(driver, 20)
    btn_csv = wait.until(expected_conditions.element_to_be_clickable((By.ID, 'aExportCsv')))
    btn_csv.click()
    
    # Espera o arquivo na pasta Downloads
    time.sleep(0.5)

    nome_final = f"resumo_{data_dt.strftime('%Y%m')}.csv"
    destino_final = os.path.join(pasta_destino_projeto, nome_final)
    lista_arquivos = glob.glob(os.path.join(pasta_downloads_padrao, "dados*.csv"))

    # Pega o primeiro arquivo dados que tiver na pasta
    arquivo = lista_arquivos[0]

    # Move o arquivo de downloads para o projeto
    shutil.move(arquivo, destino_final)
    print(f"SUCESSO: Arquivo salvo em {destino_final}")

In [5]:
# --- CONFIGURAÇÃO DO USUÁRIO ---
ANO_DESEJADO = range(2013,2022)

aba_atual = ""

for ano in ANO_DESEJADO:

    print(f"--- Iniciando robô para baixar resumos de: {ano} ---")
    if aba_atual != "antiga":
        if ano <= 2024:
            # Clica na aba das informações antigas
            aba = navegador.find_element(By.LINK_TEXT, 'Dados de 2000 a 2024')

            aba_atual = "antiga"

            aba.click()
        else:
            if aba_atual != "nova":
                # Tenta clicar na aba nova (caso esteja na antiga)
                aba = navegador.find_element(By.PARTIAL_LINK_TEXT, 'Dados a partir de 2025')

                aba_atual = "nova"

                aba.click()

    # Clica no botão de datas
    navegador.find_element(By.ID, 'btnDataBase').click()

    if ano <= 2024:
        esperar = WebDriverWait(navegador,20)
        # Usa CSS_SELECTOR para procurar por "li" DENTRO de "#ulDataBase"
        # O comando abaixo espera até que o primeiro item da lista seja visível
        esperar.until(expected_conditions.visibility_of_element_located((By.CSS_SELECTOR, "#ulDataBase li")))
        
        # Seleciona o Dropdown completo de datas
        lista_antiga = navegador.find_element(By.ID, 'ulDataBase')
    else:
        time.sleep(0.5)

    # Cria uma lista com a seção de todas as datas
    lista_elementos = navegador.find_element(By.ID, 'ulDataBase').find_elements(By.TAG_NAME, 'a')
    # List Comprehension para criar uma lista apartir do dropdown de datas
    todas_datas_texto = [i.get_attribute("innerText").strip() for i in lista_elementos]

    # Fecha o Dropdown
    navegador.find_element(By.ID, 'btnDataBase').click()

    datas_para_baixar = []

    for texto in todas_datas_texto:
        if texto == 'Selecione' or texto == '':
            continue

        # Converte texto para data para checar o ano
        dt = pd.to_datetime(texto, format='%m/%Y', errors='coerce')
        
        # Se a data for válida E o ano for igual ao pedido
        if pd.notna(dt) and dt.year == ano:
            datas_para_baixar.append(texto)

    print(f"Encontrei {len(datas_para_baixar)} relatórios para este ano: {datas_para_baixar}")

    for data_atual in datas_para_baixar:
        # Chama função para o ano selecionada
        baixar_uma_data(navegador, data_atual)

    print(f"\nSucesso! Todos os arquivos de {ano} foram baixados.")

--- Iniciando robô para baixar resumos de: 2013 ---
Encontrei 4 relatórios para este ano: ['12/2013', '09/2013', '06/2013', '03/2013']
Campo data selecionado
Aguardando lista de datas...
Data selecionada no menu.
Instituição selecionada no menu.
Relatório selecionado no menu.
Aguardando botão de download...
SUCESSO: Arquivo salvo em ../dados/bacen/resumo_bacen\resumo_201312.csv
Campo data selecionado
Aguardando lista de datas...
Data selecionada no menu.
Instituição selecionada no menu.
Relatório selecionado no menu.
Aguardando botão de download...
SUCESSO: Arquivo salvo em ../dados/bacen/resumo_bacen\resumo_201309.csv
Campo data selecionado
Aguardando lista de datas...
Data selecionada no menu.
Instituição selecionada no menu.
Relatório selecionado no menu.
Aguardando botão de download...
SUCESSO: Arquivo salvo em ../dados/bacen/resumo_bacen\resumo_201306.csv
Campo data selecionado
Aguardando lista de datas...
Data selecionada no menu.
Instituição selecionada no menu.
Relatório seleci